# Этап 8: Сравнение моделей

Сводное сравнение всех 5 моделей (Logistic Regression, CatBoost, MLP, FT-Transformer, TabM).
**Основной вопрос:** способен ли TabM превзойти CatBoost?

Ноутбук только читает артефакты из `results/` — модели не переобучаются.

## 1. Загрузка артефактов

In [ ]:
# Импортируем библиотеки для работы с данными и построения графиков
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_curve, auc  # roc_curve строит ROC, auc считает площадь под ней

# Папка с артефактами (метриками, предсказаниями, графиками)
RESULTS = Path('../results')

# Список 5 моделей: (ключ для файла, человекочитаемое название)
MODELS = [
    ('logreg',      'Logistic Regression'),   # Линейная модель-бейзлайн
    ('catboost',    'CatBoost'),              # Градиентный бустинг
    ('mlp',         'MLP'),                  # Многослойный перцептрон (нейросеть)
    ('transformer', 'FT-Transformer'),        # Трансформер для табличных данных
    ('tabm',        'TabM'),                  # Новая архитектура: ансамбль линейных голов
]

## 2. Загрузка метрик

In [ ]:
def load_metrics(model_key: str) -> dict:
    """Загрузить метрики модели в едином F2-only формате."""
    with open(RESULTS / "metrics" / f"{model_key}.json", encoding="utf-8") as f:
        raw = json.load(f)

    splits = raw.get("metrics", raw)
    test = raw.get("test_f2_metrics") or splits.get("test", {})

    return {
        "train": splits.get("train", {}),
        "val": splits.get("val", {}),
        "test": test,
        "threshold_f2": raw.get("threshold_f2") or raw.get("threshold", 0.0),
    }


all_metrics = {key: load_metrics(key) for key, _ in MODELS}
print("Metrics loaded for:", list(all_metrics.keys()))

## 3. Итоговая таблица — F2-оптимальный порог

In [ ]:
# Список метрик для отображения в итоговой таблице
METRIC_COLS = ['roc_auc', 'precision', 'recall', 'f2']
METRIC_NAMES = ['ROC-AUC', 'Precision', 'Recall', 'F2']

rows = []
for key, label in MODELS:
    m = all_metrics[key]['test']  # Берём метрики на ТЕСТОВОЙ выборке
    row = {
        'Model': label,
        # Порог классификации, подобранный по максимальному F2 на val-выборке
        'Threshold': round(all_metrics[key]['threshold_f2'], 4),
    }
    for col, name in zip(METRIC_COLS, METRIC_NAMES):
        row[name] = round(m.get(col, float('nan')), 4)
    # FP = False Positives: сколько здоровых пациентов модель ошибочно пометила как "группа риска"
    row['FP (test)'] = m.get('fp', '?')
    rows.append(row)

df_test = pd.DataFrame(rows).set_index('Model')
print('=== Test-set метрики (F2-оптимальный порог) ===')
# ROC-AUC: качество ранжирования независимо от порога (главная метрика для сравнения моделей)
# Precision: из всех "предсказанных рисковых" — какая доля реально вернулась (точность)
# Recall: из всех реально рисковых — сколько мы нашли (полнота, НЕ ПРОПУСТИТЬ важно)
# F2: взвешенное среднее precision и recall, где recall весит в 2 раза больше
# FP: абсолютное число ложных тревог — важно для операционных расходов
df_test

## 4. До и после расширенного тюнинга

In [ ]:
# Сравнение моделей ДО и ПОСЛЕ расширенного подбора гиперпараметров.
# "До" — исходный небольшой grid search (4 конфигурации).
# "После" — расширенный (13 конфигов для CatBoost, 8 для TabM).
# Цель: проверить, можно ли выжать больше из датасета.
ORIGINAL = {
    'CatBoost':  {'roc_auc': 0.6542, 'precision': 0.1100, 'recall': 0.8127, 'f2': 0.3568, 'fp': 8255},
    'TabM':      {'roc_auc': 0.6585, 'precision': 0.1136, 'recall': 0.7689, 'f2': 0.3571, 'fp': 7526},
}

ba_rows = []
for key, label in [('catboost', 'CatBoost'), ('tabm', 'TabM')]:
    orig = ORIGINAL[label]
    new  = all_metrics[key]['test']
    for version, m in [('До (4 конфига)', orig), ('После (13/8 конфигов)', new)]:
        ba_rows.append({
            'Модель': label,
            'Версия': version,
            'ROC-AUC': round(m.get('roc_auc', m.get('roc_auc', 0)), 4),
            'Precision': round(m.get('precision', 0), 4),
            'Recall': round(m.get('recall', 0), 4),
            'F2': round(m.get('f2', 0), 4),
            'FP': m.get('fp', '?'),
        })

df_ba = pd.DataFrame(ba_rows).set_index(['Модель', 'Версия'])
print('=== До/после расширенного тюнинга (test set, F2-порог) ===')
# Вывод: прирост < 0.002 — это не ошибка модели, а потолок датасета.
# Diabetes 130-US имеет известный предел предсказательной силы ~0.66-0.68 в литературе.
print('Вывод: прирост маргинальный — потолок ROC-AUC ≈ 0.66 подтверждён')
df_ba

## 5. ROC-кривые

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for key, label in MODELS:
    # Загружаем предсказания модели на тестовой выборке (y_true и y_proba)
    pred = pd.read_csv(RESULTS / 'predictions' / f'{key}_test.csv')
    # roc_curve вычисляет TPR и FPR при разных порогах
    fpr, tpr, _ = roc_curve(pred['y_true'], pred['y_proba'])
    roc_auc = auc(fpr, tpr)  # Площадь под кривой (AUC)
    ax.plot(fpr, tpr, label=f'{label} (AUC={roc_auc:.3f})', linewidth=1.5)

# Диагональная линия — случайная модель (AUC = 0.5)
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5, label='Случайная')
ax.set_xlabel('False Positive Rate')  # Доля здоровых, ошибочно помеченных как больные
ax.set_ylabel('True Positive Rate')   # Доля больных, правильно найденных (recall)
ax.set_title('ROC-кривые (тестовая выборка)')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS / 'roc_curves.png', dpi=120)
plt.show()
# Все кривые кластеризуются вокруг AUC ≈ 0.65-0.66 — это потолок датасета, а не проблема моделей.
print("Все модели кластеризуются вокруг AUC≈0.65-0.66 — потолок датасета")

## 6. ROC-кривые

## 7. Выводы и честный вердикт

### 1. Потолок ROC-AUC ≈ 0.66 — свойство датасета

Расширенный тюнинг гиперпараметров не дал значимого прироста:
- CatBoost: 4 → 13 конфигов, ROC-AUC 0.6542 → 0.6544 (+0.0002)
- TabM: 4 → 8 конфигов, ROC-AUC 0.6585 → 0.6570 (−0.0015)

SHAP-отбор признаков (RandomForest + 3-fold CV на LogReg, варианты top-10/12/15/20/25/30/all из ~37 кандидатов) подтвердил то же самое: лучший набор — top-25 с ROC-AUC 0.644 на CV, разница между вариантами < 0.002.

Это согласуется с литературой: задача 30-дневной реадмиссии на датасете Diabetes 130-US имеет известный потолок около 0.66–0.68. Причина — данные административного учёта без клинических заметок и временных рядов.

### 2. Идея с косинусным сходством — отклонена (см. `09_cosine_experiment.ipynb`)

| Показатель | Значение |
|---|---|
| Негативы с косинусом ≥ 0.6 к прототипу | ~1.6% |
| Доля всех FP, попавших в эту группу | ~2.7% |
| ROC-AUC косинуса как классификатора | 0.585 |

Гипотеза не подтвердилась: FP не концентрируются среди «косинусно похожих» примеров.

### 3. Реальный рычаг — выбор операционной точки

7 000–8 000 false positives — это не баг модели, а **прямое следствие F2-порога** (recall ×4 важнее precision при дисбалансе ~11%).

Основной протокол оставлен на F2, потому что для задачи раннего выявления важнее высокий recall и низкое число FN.


### 4. TabM vs CatBoost

На F2-оптимальном пороге (основной протокол), все модели обучены на 25 признаках (12 числовых + 13 категориальных, отобраны SHAP):

| Метрика | CatBoost | TabM | Лидер |
|---|---|---|---|
| ROC-AUC (test) | 0.6544 | **0.6570** | **TabM** (+0.003) |
| F2 (test) | **0.3602** | 0.3565 | **CatBoost** |
| Recall | **0.778** | 0.771 | **CatBoost** |
| FP | **7 552** | 7 589 | **CatBoost** |

**Вердикт:** TabM незначительно лучше по ROC-AUC, CatBoost по F2 и recall. Разница в пределах шума при разных запусках. Однозначного победителя нет — на данном датасете с 25 признаками TabM не демонстрирует явного превосходства над CatBoost.